In [1]:
import numpy as np
import pandas as pd
from scipy.stats import fit, nbinom, skewnorm

from sportstradamus.stats import StatsNBA
from sportstradamus.training.config import load_distribution_config

/home/trevor/Sportstradamus/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: 'archive'

In [ ]:
nba = StatsNBA()
nba.load()
market_stats = {}
stat_dist = load_distribution_config().get("NBA", {})

In [ ]:
for market, dist in stat_dist.items():
    try:
        if dist == "SkewNormal":
            dist_fit = (
                nba.gamelog.groupby("PLAYER_ID")[market]
                .apply(lambda x: skewnorm.fit(x.astype(int)) if len(x) > 60 else np.nan)
                .dropna()
            )
            market_stats[market] = pd.DataFrame(
                (dist_fit).to_list(), columns=["a", "loc", "scale"], index=dist_fit.index
            )
        elif dist == "NegBin":
            dist_fit = (
                nba.gamelog.groupby("PLAYER_ID")[market]
                .apply(
                    lambda x: tuple(fit(nbinom, x.astype(int), [(0, 1000), (0, 1)]).params)
                    if len(x) > 60
                    else np.nan
                )
                .dropna()
            )
            market_stats[market] = pd.DataFrame(
                (dist_fit).to_list(), columns=["n", "p", "loc"], index=dist_fit.index
            )
    except Exception as e:
        print(f"Error fitting {market}: {e}")
        continue

NameError: name 'stat_dist' is not defined